## 1. Setup Konfigurasi dan Library


In [11]:
import os, re, ast, json, random, warnings
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED       = 42
CPU_COUNT  = os.cpu_count() or 2
WORKERS    = max(1, CPU_COUNT - 1)
INPUT_DIR  = Path("/kaggle/input")
OUTPUT_DIR = Path("/kaggle/working/outputs_scenario1_lda")
LDA_K_LIST = [5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100]

random.seed(SEED)
np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for env in ["OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"]:
    os.environ[env] = str(CPU_COUNT)

print(f"CPU={CPU_COUNT} | WORKERS={WORKERS} | K candidates={LDA_K_LIST}")

CPU=4 | WORKERS=3 | K candidates=[5, 10, 15, 20, 25, 30, 40, 50, 60, 75, 100]


## 2. Load Dataset


In [12]:
def find_dataset(base):
    matches = list(base.rglob("dataset_pubmed_clean.parquet"))
    if matches: return matches[0]
    raise FileNotFoundError("dataset_pubmed_clean.parquet not found in /kaggle/input")

df = pd.read_parquet(find_dataset(INPUT_DIR))
print(f"Shape: {df.shape} | Cols: {df.columns.tolist()}")

Shape: (26046, 9) | Cols: ['pmid', 'title', 'abstract', 'raw_tags', 'clean_tags', 'final_labels', 'raw_word_count', 'preprocessed_abstract', 'clean_abstract']


## 3. Prepare Text & Labels


In [13]:
def get_col(df, names, required=False):
    m = {c.lower(): c for c in df.columns}
    for n in names:
        if n.lower() in m: return m[n.lower()]
    if required: raise ValueError(f"Column not found: {names}")
    return None

def parse_labels(v):
    if isinstance(v, (list, np.ndarray)): return list(v)
    try:
        if pd.isna(v): return []
    except (TypeError, ValueError): pass
    s = str(v).strip()
    try:
        p = ast.literal_eval(s)
        if isinstance(p, list): return [str(x) for x in p]
    except: pass
    return [x.strip() for x in re.split(r"[;,|]", s) if x.strip()]

def clean_lda(text):
    t = str(text).lower()
    t = re.sub(r"<.*?>", " ", t)
    t = re.sub(r"[^a-z\s]", " ", t)
    return re.sub(r"\s+", " ", t).strip()

col_pmid = get_col(df, ["pmid", "paper_id", "id"])
col_abs  = get_col(df, ["abstract", "clean_abstract"], required=True)
col_lbl  = get_col(df, ["final_labels"])

data = pd.DataFrame({
    "paper_id"   : df[col_pmid].astype(str) if col_pmid else np.arange(len(df)).astype(str),
    "abstract"   : df[col_abs].fillna("").astype(str),
    "label_list" : df[col_lbl].apply(parse_labels) if col_lbl else [[] for _ in range(len(df))],
})
data["primary_label"] = data["label_list"].apply(lambda x: x[0] if x else "Unknown")
data = data[data["primary_label"] != "Unknown"].reset_index(drop=True)

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    data["text_for_lda"] = list(ex.map(clean_lda, data["abstract"], chunksize=500))

data = (data[data["text_for_lda"].str.split().str.len() >= 5]
           .drop_duplicates("text_for_lda")
           .reset_index(drop=True))

print(f"Rows={len(data)} | Labels={data['primary_label'].nunique()}")

Rows=25997 | Labels=10


## 4. Build Corpus


In [14]:
from gensim.utils import simple_preprocess
from gensim.parsing.preprocessing import STOPWORDS
from gensim.corpora import Dictionary

STOP = set(STOPWORDS) | {
    "study","studies","result","results","conclusion","conclusions",
    "method","methods","background","objective","objectives",
    "patient","patients","clinical","trial","analysis","data",
    "using","use","used","based","included","including",
    "significant","significantly","associated","compared","comparison",
    "group","groups","showed","shown","observed","demonstrated"
}

def tokenize(text):
    return [t for t in simple_preprocess(text, deacc=True, min_len=3) if t not in STOP]

with ProcessPoolExecutor(max_workers=WORKERS) as ex:
    texts = list(ex.map(tokenize, data["text_for_lda"], chunksize=500))

valid = [len(t) >= 5 for t in texts]
data  = data[valid].reset_index(drop=True)
texts = [t for t, ok in zip(texts, valid) if ok]

dictionary = Dictionary(texts)
dictionary.filter_extremes(no_below=5, no_above=0.5, keep_n=50_000)
corpus = [dictionary.doc2bow(t) for t in texts]

print(f"Docs={len(corpus)} | Vocab={len(dictionary)}")

Docs=25997 | Vocab=19918


## 5. Evaluation Functions


In [15]:
from gensim.models import LdaMulticore, CoherenceModel
from sklearn.metrics import f1_score
from scipy.optimize import linear_sum_assignment

def topic_uniqueness(tw):
    all_w = [w for t in tw for w in t]
    return round(len(set(all_w)) / len(all_w), 4) if all_w else 0.0

def get_topic_words(model, topn=10):
    return [[w for w, _ in model.show_topic(i, topn=topn)] for i in range(model.num_topics)]

def predict_topics(model, corpus):
    return np.array([max(model.get_document_topics(bow, minimum_probability=0), key=lambda x: x[1])[0]
                     for bow in corpus])

def hungarian_f1(y_true, y_topic):
    labels = sorted(set(y_true))
    topics = sorted(set(y_topic))
    l2i = {l: i for i, l in enumerate(labels)}
    t2i = {t: i for i, t in enumerate(topics)}
    yt  = np.array([l2i[y] for y in y_true])
    yp  = np.array([t2i[y] for y in y_topic])
    M   = np.zeros((len(topics), len(labels)))
    for t in topics:
        for l in labels:
            M[t2i[t], l2i[l]] = f1_score((yt == l2i[l]).astype(int),
                                          (yp == t2i[t]).astype(int), zero_division=0)
    ri, ci = linear_sum_assignment(-M)
    t2l  = {topics[r]: labels[c] for r, c in zip(ri, ci)}
    fb   = pd.Series(y_true).mode()[0]
    return round(f1_score(y_true, [t2l.get(t, fb) for t in y_topic], average="macro", zero_division=0), 4)

def evaluate_lda(model):
    tw = get_topic_words(model, topn=10)
    yp = predict_topics(model, corpus)
    cv = CoherenceModel(topics=tw, texts=texts, dictionary=dictionary,
                        coherence="c_v", processes=WORKERS).get_coherence()
    return {
        "experiment": "LDA", "embedding_model": "-", "model": "LDA",
        "K": model.num_topics, "k_type": "k_search", "n_topics_found": model.num_topics,
        "outlier_rate": 0.0,
        "cv": round(cv, 4), "topic_uniqueness": topic_uniqueness(tw),
        "hungarian_f1": hungarian_f1(data["primary_label"].tolist(), yp)
    }

print("Evaluation functions ready.")

Evaluation functions ready.


## 6. Train LDA K-Search (CPU multicore)


In [16]:
results, models = [], {}

for k in LDA_K_LIST:
    print(f"Training LDA K={k} ...", end=" ", flush=True)
    model = LdaMulticore(
        corpus=corpus, id2word=dictionary, num_topics=k,
        random_state=SEED, chunksize=2000, passes=10, iterations=100,
        workers=WORKERS, alpha="asymmetric", eta="auto", eval_every=None
    )
    m = evaluate_lda(model)
    results.append(m)
    models[k] = model
    print(f"Cv={m['cv']} | F1={m['hungarian_f1']} | Uniq={m['topic_uniqueness']}")

results_df = pd.DataFrame(results)
display(results_df.sort_values("cv", ascending=False))

Training LDA K=5 ... Cv=0.5141 | F1=0.1395 | Uniq=0.8
Training LDA K=10 ... Cv=0.5809 | F1=0.2794 | Uniq=0.83
Training LDA K=15 ... Cv=0.6331 | F1=0.286 | Uniq=0.8
Training LDA K=20 ... Cv=0.5948 | F1=0.2423 | Uniq=0.76
Training LDA K=25 ... Cv=0.6017 | F1=0.2762 | Uniq=0.768
Training LDA K=30 ... Cv=0.6377 | F1=0.2656 | Uniq=0.8033
Training LDA K=40 ... Cv=0.6084 | F1=0.2289 | Uniq=0.7675
Training LDA K=50 ... Cv=0.6289 | F1=0.2382 | Uniq=0.766
Training LDA K=60 ... Cv=0.6277 | F1=0.2565 | Uniq=0.77
Training LDA K=75 ... Cv=0.6014 | F1=0.2214 | Uniq=0.7587
Training LDA K=100 ... Cv=0.5828 | F1=0.2395 | Uniq=0.756


,experiment,embedding_model,model,K,k_type,n_topics_found,outlier_rate,cv,topic_uniqueness,hungarian_f1
5,LDA,-,LDA,30,k_search,30,0.0,0.6377,0.8033,0.2656
2,LDA,-,LDA,15,k_search,15,0.0,0.6331,0.8000,0.2860
7,LDA,-,LDA,50,k_search,50,0.0,0.6289,0.7660,0.2382
8,LDA,-,LDA,60,k_search,60,0.0,0.6277,0.7700,0.2565
6,LDA,-,LDA,40,k_search,40,0.0,0.6084,0.7675,0.2289
4,LDA,-,LDA,25,k_search,25,0.0,0.6017,0.7680,0.2762
9,LDA,-,LDA,75,k_search,75,0.0,0.6014,0.7587,0.2214
3,LDA,-,LDA,20,k_search,20,0.0,0.5948,0.7600,0.2423
10,LDA,-,LDA,100,k_search,100,0.0,0.5828,0.7560,0.2395
1,LDA,-,LDA,10,k_search,10,0.0,0.5809,0.8300,0.2794


## 7. Save Output


In [17]:
topic_rows = [
    {
        "experiment": "LDA", "embedding_model": "-", "model": "LDA",
        "K": k, "k_type": "k_search", "topic_id": tid,
        "top_words": ", ".join(w for w, _ in model.show_topic(tid, topn=15)),
        "weights": json.dumps({w: round(float(s), 6) for w, s in model.show_topic(tid, topn=15)})
    }
    for k, model in models.items()
    for tid in range(k)
]

results_df.to_csv(OUTPUT_DIR / "lda_scenario1_results.csv", index=False)
pd.DataFrame(topic_rows).to_csv(OUTPUT_DIR / "lda_scenario1_topics.csv", index=False)
results_df.sort_values("cv", ascending=False).head(1).to_csv(OUTPUT_DIR / "lda_scenario1_best_by_cv.csv", index=False)
results_df.sort_values("hungarian_f1", ascending=False).head(1).to_csv(OUTPUT_DIR / "lda_scenario1_best_by_f1.csv", index=False)
data.to_csv(OUTPUT_DIR / "lda_scenario1_dataset_used.csv", index=False)

print(f"Saved to {OUTPUT_DIR}")

Saved to /kaggle/working/outputs_scenario1_lda
